# 基于 LlamaIndex 的小型 RAG 系统与 Ragas 自动化评估

本 Notebook 面向已经具备 Python 基础、正在学习 Agent/RAG 的同学。它会从零构建一个可以执行的企业制度问答系统，并用 Ragas 评估检索和回答质量。

本案例的知识库由 6 份真实 `.docx` 企业制度文件组成。Notebook 使用 LlamaIndex 的 `SimpleDirectoryReader` 和 `DocxReader` 读取文件，而不是把知识内容预先写成 JSON 或 Python 字典。唯一的外部调用是 DashScope 的 Embedding 与生成模型。

完成后你将理解这条完整链路：文档 → Embedding → VectorStoreIndex → Retriever/QueryEngine → 回答 → Golden Dataset → Ragas 指标 → 失败案例分析。


## 0. 在 macOS 中选择 `agent-base` Kernel

先在 Terminal 中确认环境：

```bash
conda activate agent-base
python -m ipykernel install --user --name agent-base --display-name "agent-base"
```

你的 `agent-base` 已安装 Jupyter Kernel，但当前没有安装 `jupyter-lab` 启动程序。你可以继续用现有的 Jupyter、VS Code 或 PyCharm 打开 Notebook，然后将 Kernel 切换为 `agent-base`。如果希望直接从该环境启动 JupyterLab，再执行：

```bash
conda install -c conda-forge jupyterlab -y
jupyter lab
```

本机已核对的核心版本为 Python 3.13、LlamaIndex Core 0.14.24 和 Ragas 0.4.3。

如果将来环境缺少依赖，可在 `agent-base` 中安装：

```bash
conda install -c conda-forge docx2txt -y
python -m pip install -U llama-index-core llama-index-readers-file llama-index-embeddings-dashscope llama-index-llms-dashscope ragas pandas rapidfuzz
```


In [1]:
import os
import sys
import warnings
from importlib.metadata import version

packages = [
    "llama-index-core",
    "llama-index-readers-file",
    "llama-index-embeddings-dashscope",
    "llama-index-llms-dashscope",
    "ragas",
    "pandas",
    "docx2txt",
]

print("Python:", sys.version.split()[0])
for package in packages:
    print(f"{package}: {version(package)}")

if not os.getenv("DASHSCOPE_API_KEY"):
    raise RuntimeError(
        "没有找到 DASHSCOPE_API_KEY。请先在 Terminal 执行 "
        "`export DASHSCOPE_API_KEY='你的Key'`，然后从该 Terminal 启动 Jupyter。"
    )

print("DASHSCOPE_API_KEY 已配置（不会打印 Key 内容）")


Python: 3.13.14
llama-index-core: 0.14.24
llama-index-readers-file: 0.6.0
llama-index-embeddings-dashscope: 0.5.0
llama-index-llms-dashscope: 0.6.1
ragas: 0.4.3
pandas: 2.3.3
docx2txt: 0.9
DASHSCOPE_API_KEY 已配置（不会打印 Key 内容）


## 1. 准备小型企业知识库

企业知识库中的 Word 文件通常没有人为编写的 `doc_id`。本案例将 `knowledge_base` 目录下的 `.docx` 文件视为原始事实来源，加载器会自动把 `file_name` 和 `file_path` 写入 LlamaIndex Document 的 Metadata。

Ragas 的 ID-based 指标只需要双方使用一致的来源标识，并不要求原文档内部真的存在 `doc_id`。因此，这里把文件名当作评估阶段的来源标签。例如，`01_员工休假制度.docx` 既是磁盘上的真实文件名，也是 Golden Dataset 指向标准来源的方式。


In [ ]:
from pathlib import Path

# 建议从 Notebook 所在目录启动 Jupyter/VS Code。
# 如果当前工作目录不同，可以手工把这里改成 knowledge_base 的绝对路径。
KNOWLEDGE_DIR = Path.cwd() / "knowledge_base"

if not KNOWLEDGE_DIR.exists():
    raise FileNotFoundError(
        f"没有找到知识库目录：{KNOWLEDGE_DIR}\n"
        "请先将当前工作目录切换到 Notebook 所在文件夹，"
        "或者手工修改 KNOWLEDGE_DIR。"
    )

docx_files = sorted(KNOWLEDGE_DIR.glob("*.docx"))
if not docx_files:
    raise FileNotFoundError(f"{KNOWLEDGE_DIR} 中没有 .docx 文件")

print(f"知识库目录：{KNOWLEDGE_DIR}")
print(f"发现 {len(docx_files)} 份 Word 文档：")
for path in docx_files:
    print("-", path.name)


## 2. 配置 DashScope LLM 与 Embedding

Embedding 模型把文档和问题转换成向量，LLM 根据召回的上下文生成最终答案。这里延续原 `Llamaindex.ipynb` 的技术路线，使用 DashScope 对接 LlamaIndex。


In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.dashscope import (
    DashScopeEmbedding,
    DashScopeTextEmbeddingModels,
    DashScopeTextEmbeddingType,
)
from llama_index.llms.dashscope import DashScope
from llama_index.readers.file import DocxReader

embed_model = DashScopeEmbedding(
    model_name=DashScopeTextEmbeddingModels.TEXT_EMBEDDING_V3,
    text_type=DashScopeTextEmbeddingType.TEXT_TYPE_DOCUMENT,
)

llm = DashScope(
    model_name="qwen-plus",
    api_key=os.environ["DASHSCOPE_API_KEY"],
    temperature=0.0,
)

Settings.embed_model = embed_model
Settings.llm = llm

print("Embedding 模型和 LLM 配置完成")


## 3. 构建 LlamaIndex 向量索引

`SimpleDirectoryReader` 负责扫描目录，`DocxReader` 负责从 Word 文件中提取文本。加载完成后，每个 LlamaIndex Document 都会带有 `file_name`、`file_path` 等 Metadata。后续 Node 会继承这些信息，因此系统能够追踪召回内容来自哪份文件，而不需要修改原始 Word 文档。


In [ ]:
documents = SimpleDirectoryReader(
    input_dir=str(KNOWLEDGE_DIR),
    required_exts=[".docx"],
    file_extractor={".docx": DocxReader()},
    recursive=False,
    raise_on_error=True,
).load_data()

print(f"LlamaIndex 实际加载了 {len(documents)} 个 Document")
for document in documents:
    print(
        document.metadata.get("file_name"),
        "| 字符数：",
        len(document.text),
    )

index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    show_progress=True,
)

retriever = index.as_retriever(similarity_top_k=2)
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=2,
    response_mode="compact",
)

print("索引构建完成，Node 数量：", len(index.docstore.docs))


## 4. 先观察一次完整 RAG 查询

一个 RAG 查询包含两个关键结果：`response` 是 LLM 生成的回答，`source_nodes` 是实际送给 LLM 的召回上下文。评估 RAG 时不能只看回答，还必须分别检查 Retriever 与 Generator。


In [ ]:
demo_question = "员工的门禁卡丢了，应该在多长时间内报告？"
demo_response = query_engine.query(demo_question)

print("问题：", demo_question)
print("回答：", str(demo_response))
print("\n召回上下文：")

for rank, source_node in enumerate(demo_response.source_nodes, start=1):
    node = source_node.node
    source_file = node.metadata.get("file_name")
    if not source_file and node.metadata.get("file_path"):
        source_file = Path(node.metadata["file_path"]).name
    print(
        f"\nTop {rank} | 来源文件={source_file} "
        f"| score={source_node.score:.4f}"
    )
    print(node.get_content())


## 5. 建立 Golden Dataset

Golden Dataset 是人工确认过的评估集。每条数据至少包含用户问题、标准答案、关键事实以及支持答案的标准来源文件。

`reference_keyword` 用于判断关键数字或事实是否出现在回答中；`reference_source_files` 保存人工确认的来源文件名。这里并没有要求 Word 文档包含 ID，而是使用真实文件名建立可核查的 Ground Truth。


In [ ]:
EVAL_SET = [
    {
        "question": "正式员工每年有多少带薪年假？",
        "reference_answer": "正式员工每个自然年度享有10个工作日的带薪年假。",
        "reference_keyword": "10个工作日",
        "reference_source_files": ["01_员工休假制度.docx"],
    },
    {
        "question": "费用发生后最晚多久要提交报销？",
        "reference_answer": "员工应在费用发生后的5个工作日内提交报销申请。",
        "reference_keyword": "5个工作日",
        "reference_source_files": ["02_费用报销制度.docx"],
    },
    {
        "question": "门禁卡丢失后必须在多久内报告？",
        "reference_answer": "门禁卡遗失后必须在30分钟内报告。",
        "reference_keyword": "30分钟",
        "reference_source_files": ["04_办公区域安全制度.docx"],
    },
    {
        "question": "P1级线上故障需要在多长时间内确认告警？",
        "reference_answer": "值班工程师必须在10分钟内确认P1级线上故障告警。",
        "reference_keyword": "10分钟",
        "reference_source_files": ["05_线上故障响应制度.docx"],
    },
    {
        "question": "员工可以在每周哪一天申请远程办公？",
        "reference_answer": "符合条件的员工可以在每周三申请远程办公。",
        "reference_keyword": "每周三",
        "reference_source_files": ["03_远程办公制度.docx"],
    },
]

print(f"Golden Dataset 包含 {len(EVAL_SET)} 个问题")


## 6. 批量运行 RAG，收集 Ragas 所需字段

每个问题只调用一次 Query Engine，并将回答、上下文文本和来源文件名保存下来。这样可以避免评估阶段重复调用生成模型，也便于逐条排查问题。


In [ ]:
def get_source_file_name(source_node) -> str:
    node = source_node.node
    file_name = node.metadata.get("file_name")
    if file_name:
        return str(file_name)

    file_path = node.metadata.get("file_path")
    if file_path:
        return Path(file_path).name

    # 正常的 SimpleDirectoryReader + DocxReader 不会走到这里。
    return "unknown_source"


rag_records = []

for number, case in enumerate(EVAL_SET, start=1):
    response = query_engine.query(case["question"])
    source_nodes = response.source_nodes

    record = {
        **case,
        "response": str(response),
        "retrieved_contexts": [
            source.node.get_content() for source in source_nodes
        ],
        "retrieved_source_files": [
            get_source_file_name(source) for source in source_nodes
        ],
    }
    rag_records.append(record)
    print(f"[{number}/{len(EVAL_SET)}] {case['question']}")
    print("回答：", record["response"])
    print("召回来源：", record["retrieved_source_files"])
    print()

print("批量 RAG 执行完成")


## 7. 使用 Ragas 进行自动化评估

本案例使用四项指标。

`IDBasedContextPrecision` 衡量召回结果中正确文档所占的比例。因为 `top_k=2` 而每个问题通常只有一份标准文档，所以即使正确文档排在第一位，Precision 也可能只有约 0.5。

`IDBasedContextRecall` 衡量标准文档是否被完整召回。`StringPresence` 检查关键事实是否出现在答案中。`NonLLMStringSimilarity` 计算生成答案与标准答案的字符相似度。

Ragas 0.4.3 当前发布包对 ID 指标存在兼容性差异：弃用提示建议从 `metrics.collections` 导入，但该版本并未在那里导出这两个类。下面使用“优先新路径、失败时回退内部模块”的写法，确保与你当前环境兼容。


In [ ]:
import pandas as pd
from ragas import SingleTurnSample
from ragas.metrics.collections import (
    NonLLMStringSimilarity,
    StringPresence,
)

try:
    from ragas.metrics.collections import (
        IDBasedContextPrecision,
        IDBasedContextRecall,
    )
except ImportError:
    from ragas.metrics._context_precision import IDBasedContextPrecision
    from ragas.metrics._context_recall import IDBasedContextRecall

context_precision_metric = IDBasedContextPrecision()
context_recall_metric = IDBasedContextRecall()
keyword_metric = StringPresence()
answer_similarity_metric = NonLLMStringSimilarity()

evaluation_rows = []

for record in rag_records:
    sample = SingleTurnSample(
        user_input=record["question"],
        response=record["response"],
        reference=record["reference_answer"],
        retrieved_contexts=record["retrieved_contexts"],
        # Ragas 的字段名叫 context_ids，但这里传入的是来源文件名。
        # 它们是加载时生成的追踪标签，不是 Word 文档自带的 ID。
        retrieved_context_ids=record["retrieved_source_files"],
        reference_context_ids=record["reference_source_files"],
    )

    id_precision = context_precision_metric.single_turn_score(sample)
    id_recall = context_recall_metric.single_turn_score(sample)
    keyword_score = keyword_metric.score(
        reference=record["reference_keyword"],
        response=record["response"],
    ).value
    answer_similarity = answer_similarity_metric.score(
        reference=record["reference_answer"],
        response=record["response"],
    ).value

    evaluation_rows.append(
        {
            "question": record["question"],
            "retrieved_files": ", ".join(record["retrieved_source_files"]),
            "response": record["response"],
            "id_context_precision": float(id_precision),
            "id_context_recall": float(id_recall),
            "keyword_presence": float(keyword_score),
            "answer_string_similarity": float(answer_similarity),
        }
    )

results_df = pd.DataFrame(evaluation_rows)
results_df.round(3)


## 8. 汇总指标并设置质量门槛

平均分便于比较不同版本，但不能代替逐条分析。企业项目通常会同时设置质量门槛，例如检索召回率必须达到 0.90，关键事实命中率必须达到 0.90。


In [ ]:
metric_columns = [
    "id_context_precision",
    "id_context_recall",
    "keyword_presence",
    "answer_string_similarity",
]

summary = results_df[metric_columns].mean().to_frame("mean_score")
display(summary.round(3))

QUALITY_GATES = {
    "id_context_recall": 0.90,
    "keyword_presence": 0.90,
}

print("质量门槛检查：")
for metric_name, threshold in QUALITY_GATES.items():
    actual = float(summary.loc[metric_name, "mean_score"])
    status = "PASS" if actual >= threshold else "FAIL"
    print(f"{status} | {metric_name}: {actual:.3f} >= {threshold:.2f}")


## 9. 自动定位失败案例

如果 `id_context_recall` 低，说明 Retriever 没有召回标准文档，应优先检查 Embedding、Chunk、Query Rewrite 或 Hybrid Search。如果 Recall 高但 `keyword_presence` 低，说明上下文已经正确，问题更可能出在 Prompt 或 Generator。


In [ ]:
failed_cases = results_df[
    (results_df["id_context_recall"] < 1.0)
    | (results_df["keyword_presence"] < 1.0)
]

if failed_cases.empty:
    print("没有发现检索遗漏或关键事实遗漏。")
else:
    print(f"发现 {len(failed_cases)} 条需要分析的案例：")
    display(
        failed_cases[
            [
                "question",
                "retrieved_files",
                "response",
                "id_context_recall",
                "keyword_presence",
            ]
        ]
    )


## 10. 可选：使用 Ragas 的 LLM Judge 指标

上面的主评估已经完整使用 Ragas，且不额外消耗 Judge 模型 Token。若希望评估 Faithfulness，可以把下面的 `RUN_LLM_JUDGE` 改为 `True`。

该代码通过 DashScope 的 OpenAI-compatible endpoint 创建 Ragas Judge。Judge 本身也可能产生误判，因此企业项目中应先抽样进行人工校准，再决定阈值。


In [ ]:
RUN_LLM_JUDGE = False

if RUN_LLM_JUDGE:
    from openai import AsyncOpenAI
    from ragas.llms import llm_factory
    from ragas.metrics.collections import Faithfulness

    judge_client = AsyncOpenAI(
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )
    judge_llm = llm_factory(
        model="qwen-plus",
        provider="openai",
        client=judge_client,
    )
    faithfulness_metric = Faithfulness(llm=judge_llm)

    faithfulness_rows = []
    for record in rag_records:
        result = faithfulness_metric.score(
            user_input=record["question"],
            response=record["response"],
            retrieved_contexts=record["retrieved_contexts"],
        )
        faithfulness_rows.append(
            {
                "question": record["question"],
                "faithfulness": result.value,
                "reason": result.reason,
            }
        )

    display(pd.DataFrame(faithfulness_rows))
else:
    print("已跳过可选 LLM Judge；主 Ragas 评估不受影响。")


## 11. 练习与下一步

你可以把 `similarity_top_k` 从 2 改成 1，观察 Context Precision 是否提高；再把一个问题改成英文缩写，例如用 `PTO` 查询年假，观察中文 Embedding 是否仍能召回正确文档。

进一步实验时，可以加入相似但无关的干扰文档，观察 Context Precision 的变化；也可以把内存向量索引替换为 Milvus，并将 `file_name`、`file_path` 等来源信息保存在 Metadata 中。进入企业级阶段后，应把 Golden Dataset 独立保存为 JSONL，并在每次修改 Embedding、Chunk、Prompt 或模型后自动运行回归评估。

最重要的结论是：RAG 评估必须拆开 Retriever 与 Generator。只观察最终回答，会掩盖“召回错误但模型碰巧答对”以及“召回正确但模型没有忠实使用上下文”这两类问题。
